## Example: NER Fine-Tuning with OpenAI Using SFT + DPO 

This notebook demonstrates how to fine-tune an NER task using **Supervised Fine-Tuning (SFT)** followed by **Directed Preference Optimization (DPO)**, based on the `tensorzero` framework and OpenAI APIs. 

### Setup

In [ ]:

#!pip install tensorzero python-dotenv pandas
import os
from dotenv import load_dotenv

load_dotenv()  # Loads the .env file containing your OPENAI_API_KEY 


### Load and Prepare Dataset for SFT

In [2]:
import pandas as pd
import json

NUM_TRAIN_DATAPOINTS = 500
NUM_VAL_DATAPOINTS = 500

# Function to load and process the NER dataset
def load_ner_dataset(path: str):
    df = pd.read_csv(path)
    df["output"] = df["output"].apply(json.loads)

    train_df = df[df["split"] == 0].sample(frac=1, random_state=0).reset_index(drop=True)
    val_df = df[df["split"] == 1].sample(frac=1, random_state=0).reset_index(drop=True)

    train_df = train_df.iloc[:NUM_TRAIN_DATAPOINTS]
    val_df = val_df.iloc[:NUM_VAL_DATAPOINTS]

    return train_df, val_df

In [11]:
def convert_to_openai_format(df: pd.DataFrame):
    # Convert a dataframe into OpenAI fin-tuning format

    formatted = []
    for _, row in df.iterrows():
        formatted.append({
            "prompt": row["input"],
            "completion": json.dumps(row["output"]) + "\n" # fine-tuning format
        })
    return formatted

In [5]:
# Save JSONL
def save_as_jsonl(data: list, filepath: str):
    with open(filepath, "w") as f:
        for item in data:
            f.write(json.dumps(item) + "\n")

In [ ]:
train_df, val_df = load_ner_dataset("data/conllpp.csv")
print(f"Number of training samples: {len(train_df)}")
print(f"Number of validation samples: {len(val_df)}")

train_sft = convert_to_openai_format(train_df)
val_sft = convert_to_openai_format(val_df)

save_as_jsonl(train_sft, "ner_train.jsonl")
save_as_jsonl(val_sft, "ner_val.jsonl")

print(train_df.head())

# Optional: preview a sample
#print("Example training sample:", train_sft[0])

### Run SFT

In [32]:
# Ensure the repo root is in the Python path
import sys
import os
import openai
import tempfile
#import recipes.supervised_fine_tuning.demonstrations.openai.openai_nb

# Add the root of the repo to the Python path
#REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))  # go up from `examples/data-extraction-ner`
#if REPO_ROOT not in sys.path:
#    sys.path.insert(0, REPO_ROOT)

from pathlib import Path

# Set your model and path
MODEL_NAME = "gpt-4o-mini-2024-07-18"  # or any other model you want
openai.api_key = os.getenv("OPENAI_API_KEY")

# Upload training and validation files
def upload_jsonl(data, purpose="fine-tune"):
    with tempfile.NamedTemporaryFile(mode="w", suffix=".jsonl", delete=False) as f:
        for item in data:
            json.dump(item, f)
            f.write("\n")
        f.flush()
        with open(f.name, "rb") as file_obj:
            file = openai.files.create(file=file_obj, purpose=purpose)
    return file.id

train_file_id = upload_jsonl(train_sft)
val_file_id = upload_jsonl(val_sft)

#fine_tune_job = openai.fine_tuning.jobs.create(
#    training_file=train_file_id,
#    validation_file=val_file_id,
#    model=MODEL_NAME
#)

# Optional: Print job info
#print("Fine-tuning job launched:", fine_tune_job["id"])

# Launch fine-tuning job
#fine_tuning_job = openai_client.fine_tuning.jobs.create(
#    training_file=train_file_id,
#    validation_file=val_file_id,
#    model="gpt-4o-mini-2024-07-18"  
#)